In [1]:
import os, json, time, math, glob, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict
device = "cuda"
EOS_ID, MASK_ID, OLD_V, NEW_V = 65354, 65355, 65355, 65356

In [2]:
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train_ids_bpe.npy", recursive=True)[0])
train_ids = np.load(os.path.join(DATA_DIR, "train_ids_bpe.npy"), mmap_mode="r")
val_ids   = np.load(os.path.join(DATA_DIR, "val_ids_bpe.npy"),   mmap_mode="r")
def get_batch(split, B, T=1024):
    data = train_ids if split == "train" else val_ids
    s = np.random.randint(0, len(data) - T - 1, size=B)
    x = np.stack([data[i:i+T] for i in s]).astype(np.int64)
    return torch.from_numpy(x).to(device)
print(f"train {len(train_ids):,} | val {len(val_ids):,}")

train 200,000,000 | val 5,000,000


In [3]:
class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

In [4]:
def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

In [5]:
def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

In [6]:
def prefix_mask(T,prefix_lens,device=None):
    causal=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
    pos=torch.arange(T,device=device)
    in_prefix=pos[None,:]<prefix_lens[:,None]
    both=in_prefix[:,:,None] & in_prefix[:,None,:]
    return (causal[None] | both)[:, None]
m = prefix_mask(6, torch.tensor([3, 0, 6]))
print(tuple(m.shape)); print(m[0, 0].int())
print(torch.equal(m[1, 0], torch.tril(torch.ones(6, 6, dtype=torch.bool))), bool(m[2, 0].all()), not bool(m[0, 0, 3, 4]))

(3, 1, 6, 6)
tensor([[1, 1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1]], dtype=torch.int32)
True True True


In [7]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x,mask=None):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        if mask is None:
            mask=self.causal_mask[:T,:T]
        scores=scores.masked_fill(~mask,float("-inf"))
        
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

In [8]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

In [9]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx,prefix_lens=None):
        if prefix_lens is not None:
            mask=prefix_mask(idx.size(1),prefix_lens,idx.device)
        else:
            mask=None
        x=self.tok_emb(idx)
        for block in self.blocks: x=block(x,mask)
        x=self.norm_f(x)
        return self.head(x)

In [10]:
paths = glob.glob("/kaggle/input/**/arch3_ckpt.pt", recursive=True)
ck = torch.load(paths[0], map_location="cpu")

cfg = LlamaConfig(**ck["cfg"])
model = PrefixLM(cfg)
model.load_state_dict(ck["model"])

new_embedding = nn.Embedding(NEW_V, cfg.d_model)
with torch.no_grad():
    new_embedding.weight[:OLD_V] = model.tok_emb.weight
    new_embedding.weight[MASK_ID] = model.tok_emb.weight.mean(dim=0)

model.tok_emb = new_embedding
model.head = nn.Linear(cfg.d_model, NEW_V, bias=False)
model.head.weight = model.tok_emb.weight

cfg.vocab_size = NEW_V
model.to(device)

print(ck.get("step"), ck.get("best_val"))
del ck

399 3.585148334503174


In [11]:
print(f"{sum(p.numel() for p in model.parameters()):,} | expected 184,910,848")
print("head tied        :", model.head.weight.data_ptr() == model.tok_emb.weight.data_ptr())
print("old rows intact  :", torch.allclose(model.tok_emb.weight[:OLD_V].cpu(), torch.load(glob.glob("/kaggle/input/**/arch3_ckpt.pt", recursive=True)[0], map_location="cpu")["model"]["tok_emb.weight"]))
x = get_batch("val", 2, 64); T = x.size(1)
pl = torch.full((2,), T, device=device)
m = prefix_mask(T, pl, device)
print("fully bidirectional:", bool(m.all()))

184,910,848 | expected 184,910,848
head tied        : True
old rows intact  : True
fully bidirectional: True


In [12]:
def mlm_mask(x, p=0.15):
    labels = x.clone()
    sel = (torch.rand(x.shape, device=x.device) < p) & (x != EOS_ID)
    labels[~sel] = -100
    r = torch.rand(x.shape, device=x.device); x = x.clone()
    x[sel & (r < 0.8)] = MASK_ID
    rnd = sel & (r >= 0.8) & (r < 0.9)
    x[rnd] = torch.randint(0, EOS_ID, (int(rnd.sum()),), device=x.device)
    return x, labels

@torch.no_grad()
def mlm_val_loss(n_batches=10, B=4, T=1024):
    model.eval(); tot = 0.0
    g = torch.Generator(device=device).manual_seed(0)
    for _ in range(n_batches):
        x = get_batch("val", B, T); xm, lab = mlm_mask(x)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(xm, torch.full((B,), T, device=device))
        tot += F.cross_entropy(lg.float().view(-1, NEW_V), lab.view(-1), ignore_index=-100).item()
    model.train(); return tot / n_batches

print("MLM val loss BEFORE training:", round(mlm_val_loss(), 3))

MLM val loss BEFORE training: 7.641


In [13]:
MICRO_B, ACCUM, T = 4, 122, 1024     # tokens/step = 4 × ACCUM × 1024
MAX_STEPS = 1000                      # sized so the run ends inside your hour budget
PEAK_LR, MIN_LR, WARMUP = 1e-4, 1e-5, 20
WEIGHT_DECAY, CLIP = 0.1, 1.0
EVAL_EVERY, SAVE_EVERY_MIN = 25, 60
TIME_LIMIT = 11* 3600              # your budget minus ~20 min
CKPT = "/kaggle/working/arch3_mlm_ckpt.pt"; LAST = "/kaggle/working/arch3_mlm_last.pt"

In [ ]:
import glob

decay = [p for _, p in model.named_parameters() if p.dim() >= 2]
nodecay = [p for _, p in model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                         {"params": nodecay, "weight_decay": 0.0}], lr=PEAK_LR)
scaler = torch.amp.GradScaler("cuda")

def lr_at(s):                                   # depends only on the GLOBAL step → continues correctly after resume
    if s < WARMUP: return PEAK_LR * (s + 1) / WARMUP
    r = (s - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * min(1.0, r)))

BEST = "/kaggle/working/arch3_mlm_best.pt"      # model only (~0.75 GB) → for fine-tuning
LAST = "/kaggle/working/arch3_mlm_last.pt"      # model + optimizer + scaler (~2.2 GB) → for resuming

def save_best(step, best):
    torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "step": step,
                "best_val": best, "mask_id": MASK_ID}, BEST)
    print(f"  saved BEST @ step {step} (val {best:.3f})")

def save_last(step, best, hist):
    torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                "cfg": asdict(cfg), "step": step, "best_val": best, "hist": hist, "mask_id": MASK_ID}, LAST)
    print(f"  saved LAST @ step {step}")

# ---------- resume if a previous run's LAST checkpoint is attached as input ----------
start, best, hist = 0, float("inf"), {"step": [], "val": []}
prev = glob.glob("/kaggle/input/**/arch3_mlm_last.pt", recursive=True)
if prev:
    ck = torch.load(prev[0], map_location="cpu")
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
    start, best, hist = ck["step"] + 1, ck["best_val"], ck["hist"]
    print(f"RESUMED from {prev[0]} | next step {start} | best val {best:.3f}")
    del ck
else:
    print("fresh start (no previous checkpoint attached)")

t0 = t_save = t_log = time.time(); model.train()
step = start - 1
for step in range(start, MAX_STEPS):
    for g_ in opt.param_groups: g_["lr"] = lr_at(step)
    for _ in range(ACCUM):
        xm, lab = mlm_mask(get_batch("train", MICRO_B, T))
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(xm, torch.full((MICRO_B,), T, device=device))
            loss = F.cross_entropy(lg.view(-1, NEW_V), lab.view(-1), ignore_index=-100) / ACCUM
        scaler.scale(loss).backward()
    scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)

    lv = loss.item() * ACCUM
    if not math.isfinite(lv): print("nan at", step); break
    if step % 5 == 0:
        torch.cuda.synchronize(); dt = time.time() - t_log; t_log = time.time()
        n = 5 if step > start else 1
        print(f"step {step:5d} | mlm loss {lv:.3f} | lr {lr_at(step):.2e} | gnorm {gn:.2f} | "
              f"{MICRO_B*ACCUM*T*n/dt:,.0f} tok/s")
    if step > start and step % EVAL_EVERY == 0:
        v = mlm_val_loss(); hist["step"].append(step); hist["val"].append(v)
        print(f"  EVAL step {step} | MLM val {v:.3f}")
        if v < best: best = v; save_best(step, best)
    if time.time() - t_save > SAVE_EVERY_MIN * 60:
        save_last(step, best, hist); t_save = time.time()
    if time.time() - t0 > TIME_LIMIT:
        print(f"time limit reached at step {step}"); break

save_last(step, best, hist)
json.dump(hist, open("/kaggle/working/arch3_mlm_history.json", "w"))
print(f"this run: {(time.time()-t0)/3600:.2f} h | reached step {step} of {MAX_STEPS} | best MLM val {best:.3f}")